# Time Series Analysis — Real Datasets
**Datasets:** Sunspots (AR), CO2 (ARIMA), Airline Passengers (SARIMA), Temperature Capstone

> Note: Plotting is kept minimal in this notebook. Focus is on model fitting and results.

### Keywords:
- **statsmodels.datasets**: Built-in datasets for statistical modelling practice
- **co2**: Monthly CO2 concentration data from Mauna Loa Observatory (1958–2001)
- **sunspots**: Yearly sunspot count data (1700–2008), classic AR dataset
- **airline passengers**: Classic monthly international airline passengers dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import statsmodels.api as sm
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from sklearn.metrics import mean_absolute_error, mean_squared_error

def evaluate(actual, predicted, model_name):
    mae  = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100
    print(f"{model_name}: MAE={mae:.2f} | RMSE={rmse:.2f} | MAPE={mape:.2f}%")
    return {'MAE': round(mae,2), 'RMSE': round(rmse,2), 'MAPE': round(mape,2)}

def adf_test(series, name=''):
    p = adfuller(series.dropna())[1]
    status = 'STATIONARY ✓' if p < 0.05 else 'NON-STATIONARY ✗'
    print(f"ADF [{name}]: p={p:.4f} → {status}")
    return p < 0.05

print("Libraries loaded.")

---
## Part 1: AR Model — Sunspots Dataset

**Dataset:** `statsmodels.datasets.sunspots` — yearly sunspot counts from 1700 to 2008 (308 years).

**Why AR?** Sunspot activity follows a ~11-year cycle driven by solar physics — purely autoregressive, no external inputs.

### Steps:
1. Load data → check stationarity
2. Use ACF/PACF to choose p
3. Fit AutoReg → evaluate

In [ ]:
# Load sunspots dataset
sunspots = sm.datasets.sunspots.load_pandas().data
sunspots.set_index('YEAR', inplace=True)
sunspots.index = pd.to_datetime(sunspots.index, format='%Y')
sunspots.index.freq = 'YS'
series_sun = sunspots['SUNACTIVITY']

print("Shape:", series_sun.shape)
print(series_sun.describe().round(2))
adf_test(series_sun, 'Sunspots')

In [ ]:
# Quick plot
series_sun.plot(figsize=(13, 3), title='Annual Sunspot Activity (1700–2008)',
                color='darkorange')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# ACF/PACF to choose p
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
plot_acf(series_sun, lags=30, ax=axes[0], title='ACF')
plot_pacf(series_sun, lags=30, ax=axes[1], title='PACF', method='ywm')
plt.tight_layout(); plt.show()
print("PACF drops sharply at lag ~2-3 → try AR(2) and AR(9)")

In [ ]:
# Train-test split (last 20 years = test)
split = len(series_sun) - 20
train_sun = series_sun[:split]
test_sun  = series_sun[split:]

# Compare AR(2) and AR(9)
results_sun = {}
for p in [2, 9]:
    model = AutoReg(train_sun, lags=p, trend='c')
    fit   = model.fit()
    pred  = fit.predict(start=split, end=split+len(test_sun)-1)
    pred.index = test_sun.index
    results_sun[f'AR({p})'] = evaluate(test_sun, pred, f'AR({p})')

best_p = min(results_sun, key=lambda k: results_sun[k]['RMSE'])
print(f"\nBest model: {best_p}")

In [ ]:
# Plot best AR model
p_best = int(best_p.replace('AR(','').replace(')',''))
ar_sun = AutoReg(train_sun, lags=p_best, trend='c').fit()
pred_sun = ar_sun.predict(start=split, end=split+len(test_sun)-1)
pred_sun.index = test_sun.index

plt.figure(figsize=(12, 4))
plt.plot(train_sun.index[-30:], train_sun[-30:], label='Train', color='steelblue')
plt.plot(test_sun, label='Actual', color='green')
plt.plot(pred_sun, 'r--', label=f'{best_p} Forecast')
plt.title(f'Sunspots — {best_p} Forecast vs Actual')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

---
## Part 2: ARIMA Model — CO2 Dataset

**Dataset:** `statsmodels.datasets.co2` — monthly atmospheric CO2 (ppm) from Mauna Loa Observatory, 1958–2001.

**Why ARIMA?** Strong upward trend (d=1 needed) + some autocorrelation. We'll handle the seasonal component with differencing first to focus on pure ARIMA.

In [ ]:
# Load CO2 data
co2 = sm.datasets.co2.load_pandas().data
co2 = co2.fillna(method='ffill')  # forward-fill missing values
series_co2 = co2['co2']

print("Shape:", series_co2.shape)
print("Period:", series_co2.index[0], "→", series_co2.index[-1])
print(f"CO2 range: {series_co2.min():.1f} → {series_co2.max():.1f} ppm")

adf_test(series_co2, 'CO2 original')
adf_test(series_co2.diff().dropna(), 'CO2 1st diff')

In [ ]:
# Plot CO2
series_co2.plot(figsize=(13, 3), title='Monthly CO2 (ppm) — Mauna Loa 1958–2001', color='green')
plt.ylabel('CO2 (ppm)'); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# Train-test split (last 2 years = test)
split_co2 = len(series_co2) - 24
train_co2 = series_co2[:split_co2]
test_co2  = series_co2[split_co2:]

results_co2 = {}
# Try a few ARIMA orders
for p, d, q in [(1,1,1), (2,1,1), (1,1,2), (2,1,2)]:
    try:
        fit = ARIMA(train_co2, order=(p,d,q)).fit()
        pred = fit.get_forecast(steps=len(test_co2)).predicted_mean
        pred.index = test_co2.index
        label = f'ARIMA({p},{d},{q})'
        metrics = evaluate(test_co2, pred, label)
        metrics['AIC'] = round(fit.aic, 1)
        results_co2[label] = metrics
    except Exception as e:
        print(f"ARIMA({p},{d},{q}) failed: {e}")

best_co2 = min(results_co2, key=lambda k: results_co2[k]['RMSE'])
print(f"\nBest model: {best_co2} | AIC: {results_co2[best_co2]['AIC']}")

In [ ]:
# Fit best ARIMA and plot
p,d,q = [int(x) for x in best_co2.replace('ARIMA(','').replace(')','').split(',')]
arima_co2 = ARIMA(train_co2, order=(p,d,q)).fit()
fc = arima_co2.get_forecast(steps=len(test_co2))
pred_co2 = fc.predicted_mean
pred_co2.index = test_co2.index
ci_co2 = fc.conf_int()
ci_co2.index = test_co2.index

plt.figure(figsize=(12, 4))
plt.plot(train_co2.index[-48:], train_co2[-48:], label='Train', color='steelblue')
plt.plot(test_co2, label='Actual', color='green', linewidth=2)
plt.plot(pred_co2, 'r--', label=f'{best_co2} Forecast', linewidth=2)
plt.fill_between(test_co2.index, ci_co2.iloc[:,0], ci_co2.iloc[:,1], alpha=0.15, color='red')
plt.title(f'CO2 — {best_co2} Forecast')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

---
## Part 3: SARIMA Model — Airline Passengers Dataset

**Dataset:** Classic airline passengers dataset — monthly international airline passengers (1949–1960, 144 months).

**Why SARIMA?** Strong trend (d=1) + multiplicative seasonal pattern growing with trend (log transform → D=1, m=12).

In [ ]:
# Load airline passengers via statsmodels
from statsmodels.datasets import get_rdataset
try:
    airline = get_rdataset('AirPassengers').data
    airline.columns = ['time', 'passengers']
    dates_air = pd.date_range('1949-01', periods=144, freq='MS')
    series_air = pd.Series(airline['passengers'].values, index=dates_air, name='passengers')
except:
    # Fallback: recreate from known values
    passengers = [112,118,132,129,121,135,148,148,136,119,104,118,
                  115,126,141,135,125,149,170,170,158,133,114,140,
                  145,150,178,163,172,178,199,199,184,162,146,166,
                  171,180,193,181,183,218,230,242,209,191,172,194,
                  196,196,236,235,229,243,264,272,237,211,180,201,
                  204,188,235,227,234,264,302,293,259,229,203,229,
                  242,233,267,269,270,315,364,347,312,274,237,278,
                  284,277,317,313,318,374,413,405,355,306,271,306,
                  315,301,356,348,355,422,465,467,404,347,305,336,
                  340,318,362,348,363,435,491,505,404,359,310,337,
                  360,342,406,396,420,472,548,559,463,407,362,405,
                  417,391,419,461,472,535,622,606,508,461,390,432]
    dates_air = pd.date_range('1949-01', periods=144, freq='MS')
    series_air = pd.Series(passengers, index=dates_air, name='passengers')

print("Shape:", series_air.shape)
print(f"Range: {series_air.index[0].date()} → {series_air.index[-1].date()}")
print(f"Passengers: min={series_air.min()}, max={series_air.max()}")

In [ ]:
# Decomposition
decomp_air = seasonal_decompose(series_air, model='multiplicative', period=12)
decomp_air.plot()
plt.suptitle('Airline Passengers — Multiplicative Decomposition', y=1.01)
plt.tight_layout(); plt.show()

# Log transform to stabilize variance (multiplicative → additive)
log_air = np.log(series_air)
adf_test(log_air.diff().dropna(), 'Log(Passengers) 1st diff')

In [ ]:
# Train-test: last 12 months = test
split_air = len(series_air) - 12
train_air = log_air[:split_air]
test_air  = log_air[split_air:]

# Fit SARIMA(1,1,1)(1,1,1,12) on log data
sarima_air = SARIMAX(train_air, order=(1,1,1), seasonal_order=(1,1,1,12), trend='n')
sarima_air_fit = sarima_air.fit(disp=False)
print(f"SARIMA: AIC={sarima_air_fit.aic:.2f} | BIC={sarima_air_fit.bic:.2f}")

# Forecast (still on log scale)
fc_air = sarima_air_fit.get_forecast(steps=12)
pred_log = fc_air.predicted_mean
pred_log.index = test_air.index
ci_log = fc_air.conf_int()
ci_log.index = test_air.index

# Back-transform to original scale
pred_actual = np.exp(pred_log)
ci_actual   = np.exp(ci_log)
test_actual = np.exp(test_air)

evaluate(test_actual, pred_actual, 'SARIMA(1,1,1)(1,1,1,12)')

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(series_air.index[-36:], series_air[-36:], label='Historical', color='steelblue')
plt.plot(test_actual.index, test_actual, label='Actual', color='green', linewidth=2)
plt.plot(pred_actual.index, pred_actual, 'r--', label='SARIMA Forecast', linewidth=2)
plt.fill_between(pred_actual.index, ci_actual.iloc[:,0], ci_actual.iloc[:,1],
                  alpha=0.2, color='red', label='95% CI')
plt.title('Airline Passengers — SARIMA(1,1,1)(1,1,1,12)')
plt.ylabel('Passengers'); plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

---
## Capstone Project — Monthly Temperature Forecasting

**Task:** Forecast city temperature for the next 6 months using 10 years of monthly data.

**Approach:** Generate realistic synthetic monthly temperature data (with yearly season + warming trend),
apply the full pipeline, and forecast ahead.

> Plotting is kept minimal in this capstone — focus on the results table.

In [ ]:
np.random.seed(7)
n_temp = 120  # 10 years monthly
dates_temp = pd.date_range('2013-01', periods=n_temp, freq='MS')
t_temp = np.arange(n_temp)

# City temperature: seasonal + slight warming trend
temp = (22 + 0.03*t_temp                      # warming trend
        + 12 * np.sin(2*np.pi*(t_temp-3)/12)  # seasonal (peak in June)
        + np.random.normal(0, 1.5, n_temp))    # noise

temp_ts = pd.Series(temp, index=dates_temp, name='temp_C')

print("Temperature Data:")
print(f"  Period: {dates_temp[0].date()} to {dates_temp[-1].date()}")
print(f"  Mean: {temp_ts.mean():.1f}°C | Min: {temp_ts.min():.1f}°C | Max: {temp_ts.max():.1f}°C")

# Check stationarity
adf_test(temp_ts, 'Temp original')
adf_test(temp_ts.diff().dropna(), 'Temp 1st diff')

In [ ]:
# Train-test split
split_t = len(temp_ts) - 12
train_t = temp_ts[:split_t]
test_t  = temp_ts[split_t:]

# Fit SARIMA — best for seasonal temperature data
sarima_t = SARIMAX(train_t, order=(1,1,1), seasonal_order=(1,1,1,12), trend='c').fit(disp=False)
pred_t = sarima_t.get_forecast(steps=12).predicted_mean
pred_t.index = test_t.index

evaluate(test_t, pred_t, 'SARIMA Temperature')

# Quick test vs actual chart
plt.figure(figsize=(11, 3))
plt.plot(test_t, label='Actual', color='steelblue')
plt.plot(pred_t, 'r--', label='Forecast')
plt.title('Temperature — Test Set Forecast'); plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Forecast next 6 months on full data
full_t = SARIMAX(temp_ts, order=(1,1,1), seasonal_order=(1,1,1,12), trend='c').fit(disp=False)
future_fc = full_t.get_forecast(steps=6)
future_t = future_fc.predicted_mean
future_ci_t = future_fc.conf_int()
future_t.index = pd.date_range(temp_ts.index[-1]+pd.DateOffset(months=1), periods=6, freq='MS')
future_ci_t.index = future_t.index

# Results table — no extra plotting
print("\n6-Month Temperature Forecast:")
print("-" * 50)
print(f"{'Month':<12} {'Forecast':>10} {'Lower':>10} {'Upper':>10}")
print("-" * 50)
for date, fc, lo, hi in zip(future_t.index,
                              future_t.values,
                              future_ci_t.iloc[:,0].values,
                              future_ci_t.iloc[:,1].values):
    print(f"{date.strftime('%b %Y'):<12} {fc:>9.1f}°C {lo:>9.1f}°C {hi:>9.1f}°C")
print("-" * 50)

In [ ]:
# Final summary plot
plt.figure(figsize=(12, 4))
plt.plot(temp_ts.index[-24:], temp_ts[-24:], label='Historical', color='steelblue')
plt.plot(future_t.index, future_t, 'ro--', label='6M Forecast', linewidth=2)
plt.fill_between(future_t.index, future_ci_t.iloc[:,0], future_ci_t.iloc[:,1],
                  alpha=0.2, color='red', label='95% CI')
plt.axvline(temp_ts.index[-1], linestyle='--', color='gray', alpha=0.6)
plt.title('City Temperature — 6-Month SARIMA Forecast')
plt.ylabel('Temperature (°C)'); plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print("Capstone complete. SARIMA is best for data with trend + seasonality.")

---
## Summary — Model Selection Guide

| Model | Full Name | Use When | sklearn/statsmodels |
|-------|-----------|----------|---------------------|
| **AR(p)** | AutoRegressive | Stationary, no seasonality, past values predict | `AutoReg` |
| **MA(q)** | Moving Average | Short-term shocks influence future | `ARIMA(0,0,q)` |
| **ARIMA(p,d,q)** | AR + I + MA | Trend present (d≥1), no strong seasonality | `ARIMA` |
| **SARIMA(p,d,q)(P,D,Q,m)** | Seasonal ARIMA | Trend + seasonal patterns | `SARIMAX` |

### Quick Selection:
- **Seasonal ACF spikes at lag 12/24?** → SARIMA
- **Trend but no seasonality?** → ARIMA
- **Stationary, AR pattern in PACF?** → AR
- **Stationary, MA pattern in ACF?** → MA